In [0]:
# Cell 1: Exportação da Gold para CSV
catalog = "visagio_atividade_v_energy_360"
gold_schema = f"{catalog}.gold"
export_path = f"/Volumes/{catalog}/gold/exports"

# 1. CORREÇÃO: Recuperar os DataFrames guardados pela camada Gold
df_dim_tempo = spark.table(f"{gold_schema}.dim_tempo")
df_dim_unidade_consumidora = spark.table(f"{gold_schema}.dim_unidade_consumidora")
df_fato_medicao = spark.table(f"{gold_schema}.fato_medicao")
df_fato_anomalia_leitura = spark.table(f"{gold_schema}.fato_anomalia_leitura")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {gold_schema}.exports")

def export_gold_csv(dataframe, table_name):
    staging_path = f"{export_path}/_staging_{table_name}"
    target_path = f"{export_path}/{table_name}.csv"

    # Exporta para uma pasta temporária (staging)
    dataframe.coalesce(1).write.mode("overwrite").option("header", True).csv(staging_path)

    # Localiza o ficheiro gerado pelo Spark
    part_file = next(
        file_info.path
        for file_info in dbutils.fs.ls(staging_path)
        if file_info.name.startswith("part-") and file_info.name.endswith(".csv")
    )
    
    # 2. CORREÇÃO: Prevenir erro se for a primeira vez e o ficheiro ainda não existir
    try:
        dbutils.fs.rm(target_path)
    except Exception:
        pass 
        
    dbutils.fs.cp(part_file, target_path)
    dbutils.fs.rm(staging_path, True)
    print(f"{table_name}: CSV exportado em {target_path}")

export_gold_csv(df_dim_tempo, "dim_tempo")
export_gold_csv(df_dim_unidade_consumidora, "dim_unidade_consumidora")
export_gold_csv(df_fato_medicao, "fato_medicao")
export_gold_csv(df_fato_anomalia_leitura, "fato_anomalia_leitura")

In [0]:
# Cell 2: Exportação para SQLite (DDL do contrato, PK/FK, WAL)
import os
import shutil
import sqlite3
from datetime import date, datetime

local_db_path = "/tmp/v_energy.db"
volume_db_path = f"/Volumes/{catalog}/gold/exports/v_energy.db"
BATCH_SIZE = 50_000

# Limpar versões locais anteriores, incluindo ficheiros temporários do WAL
for suffix in ("", "-wal", "-shm"):
    if os.path.exists(local_db_path + suffix):
        os.remove(local_db_path + suffix)

connection = sqlite3.connect(local_db_path)
connection.execute("PRAGMA journal_mode=WAL;")
connection.execute("PRAGMA synchronous=NORMAL;")
connection.execute("PRAGMA foreign_keys=ON;")

# DDL das Tabelas
connection.execute("""
    CREATE TABLE dim_unidade_consumidora (
        id_uc TEXT PRIMARY KEY,
        codigo_cliente TEXT,
        nome_titular TEXT,
        logradouro TEXT,
        numero TEXT,
        bairro TEXT,
        municipio TEXT,
        uf TEXT,
        cep TEXT,
        classe_consumo TEXT,
        subgrupo_tarifario TEXT,
        alimentador TEXT,
        subestacao TEXT,
        numero_medidor_ativo TEXT,
        possui_gd BOOLEAN,
        potencia_gd_kw REAL,
        data_conexao_gd TEXT,
        status TEXT
    )""")

connection.execute("""
    CREATE TABLE fato_medicao (
        id_medicao TEXT PRIMARY KEY,
        id_uc TEXT NOT NULL,
        id_medidor TEXT,
        competencia TEXT NOT NULL,
        data_leitura TEXT,
        leitura_anterior REAL,
        leitura_atual REAL,
        consumo_kwh REAL,
        dias_faturamento INTEGER,
        media_diaria_kwh REAL,
        tipo_leitura TEXT,
        FOREIGN KEY (id_uc) REFERENCES dim_unidade_consumidora(id_uc)
    )""")

connection.execute("""
    CREATE TABLE dim_tempo (
        ano INTEGER,
        mes INTEGER,
        competencia TEXT PRIMARY KEY
    )""")

connection.execute("""
    CREATE TABLE fato_anomalia_leitura (
        id_anomalia TEXT PRIMARY KEY,
        id_uc TEXT NOT NULL,
        id_medidor TEXT,
        competencia TEXT NOT NULL,
        data_leitura TEXT,
        leitura_anterior REAL,
        leitura_atual REAL,
        diferenca_leitura_kwh REAL,
        tipo_leitura TEXT,
        motivo_anomalia TEXT,
        FOREIGN KEY (id_uc) REFERENCES dim_unidade_consumidora(id_uc)
    )""")


def to_sqlite_value(cell_value):
    # 3. CORREÇÃO: Tratamento rigoroso de nulos para não quebrar o insert
    if cell_value is None:
        return None
    if isinstance(cell_value, bool):
        return int(cell_value)
    if isinstance(cell_value, (date, datetime)):
        return cell_value.isoformat()
    return cell_value

def load_dataframe(dataframe, table_name):
    # Garante a mesma ordem de colunas da tabela criada acima
    table_columns = [row[1] for row in connection.execute(f"PRAGMA table_info({table_name})")]
    placeholders = ", ".join("?" for _ in table_columns)
    insert_sql = f"INSERT INTO {table_name} ({', '.join(table_columns)}) VALUES ({placeholders})"

    total_rows, batch = 0, []
    # Usando toLocalIterator para não estourar a memória (excelente prática no Databricks)
    for spark_row in dataframe.select(*table_columns).toLocalIterator():
        batch.append(tuple(to_sqlite_value(cell_value) for cell_value in spark_row))
        if len(batch) >= BATCH_SIZE:
            connection.executemany(insert_sql, batch)
            total_rows += len(batch)
            batch = []
            
    if batch:
        connection.executemany(insert_sql, batch)
        total_rows += len(batch)
    print(f"{table_name}: {total_rows} linhas carregadas")

# A dimensão DEVE ser carregada primeiro para não violar as chaves estrangeiras (FK)
load_dataframe(df_dim_unidade_consumidora, "dim_unidade_consumidora")  
load_dataframe(df_dim_tempo, "dim_tempo")
load_dataframe(df_fato_medicao, "fato_medicao")
load_dataframe(df_fato_anomalia_leitura, "fato_anomalia_leitura")

# Criação dos índices de alta performance exigidos para a fatia
for index_statement in [
    "CREATE INDEX IF NOT EXISTS idx_dim_uc_id ON dim_unidade_consumidora(id_uc);",
    "CREATE INDEX IF NOT EXISTS idx_dim_uc_bairro ON dim_unidade_consumidora(bairro);",
    "CREATE INDEX IF NOT EXISTS idx_dim_uc_mun ON dim_unidade_consumidora(municipio);",
    "CREATE INDEX IF NOT EXISTS idx_fato_med_uc ON fato_medicao(id_uc);",
    "CREATE INDEX IF NOT EXISTS idx_fato_med_comp ON fato_medicao(competencia);",
    "CREATE INDEX IF NOT EXISTS idx_anom_uc ON fato_anomalia_leitura(id_uc);"
]:
    connection.execute(index_statement)

connection.commit()

fk_violations = connection.execute("PRAGMA foreign_key_check;").fetchall()
if len(fk_violations) > 0:
    print(f"⚠️ AVISO: Foram encontradas {len(fk_violations)} violações de Foreign Key!")
else:
    print("✅ Integridade relacional (Foreign Keys) validada com sucesso!")

# Forçar a gravação de todas as transações WAL no ficheiro principal antes de copiar
connection.execute("PRAGMA wal_checkpoint(TRUNCATE);")
connection.close()

shutil.copy(local_db_path, volume_db_path)
print(f"✅ Banco de dados exportado com sucesso para o Volume do Unity Catalog: {volume_db_path}")